In [8]:
import numpy as np
import pandas as pd

import glob 

In [9]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
def remove_negative_fluxunc(df):
    df = df[df['forcediffimfluxunc'] > 0]
    return df

def add_mjd(df):
    df['mjd'] = df['jd'] - 2400000.5
    return df

def add_detected(df):
    df['detected'] = np.abs(df['forcediffimflux']/df['forcediffimfluxunc']) > 5
    return df

def limit_mjd(df, n_days_before, df_det):
    def approx_gte(xarray, yval):
        return (xarray >= yval) | np.isclose(xarray, yval)
    def approx_lte(xarray, yval):
        return (xarray <= yval) | np.isclose(xarray, yval)
    
    # Remove epochs outside chosen limits in mjd
    sorted_detections = df[df['detected']].sort_values('mjd')

    mask = approx_gte(sorted_detections['mjd'], df_det['firstmjd']) \
        & approx_lte(sorted_detections['mjd'], df_det['lastmjd'])
    sorted_detections = sorted_detections[mask].copy()

    if sorted_detections.empty:
        return sorted_detections

    first_detection_mjd = sorted_detections.iloc[0]['mjd']
    last_detection_mjd = sorted_detections.iloc[-1]['mjd']

    mask = (df['mjd'] > first_detection_mjd - n_days_before) & (df['mjd'] <= last_detection_mjd)
    df = df[mask]

    return df

def add_props(df):
    # Compute extra quantities
    # From https://web.ipac.caltech.edu/staff/fmasci/ztf/forcedphot.pdf page 15
    # and https://github.com/alercebroker/alerce_ztf_forced/blob/main/lib/alerceforced.py
    
    # print('Compute more quantities (total fluxes, difference magnitudes, etc)')
    
    df = df.copy()

    DISTANCE_THRESHOLD = 1.4
    
    df['nearestrefflux'] = 10.**(0.4 * (df['zpdiff'] - df['nearestrefmag']))
    df['flux_tot'] =  df['forcediffimflux'] + df['nearestrefflux']
    
    flux2uJy = 10.**((8.9 - df['zpdiff']) / 2.5) * 1.e6
    df['flux_diff_uJy'] = df['forcediffimflux'] * flux2uJy
    df['sigma_flux_diff_uJy'] = df['forcediffimfluxunc'] * flux2uJy
    df['flux_tot_uJy'] = df['flux_tot'] * flux2uJy
    
    # Note that this quantity is computed in a different way in
    # https://web.ipac.caltech.edu/staff/fmasci/ztf/forcedphot.pdf page 15,
    # Depending on nearestreffluxunc
    df['fluxunc_tot'] = df['forcediffimfluxunc']
    
    df['SNR_tot'] = df['flux_tot'] / df['fluxunc_tot']
    df['fluxunc_tot_uJy'] = df['fluxunc_tot'] * flux2uJy
    
    # Note that these quantities are computed in a different way in
    # https://web.ipac.caltech.edu/staff/fmasci/ztf/forcedphot.pdf pages 14-15,
    # Depending on a S/N threshold
    df['mag_tot'] = df['zpdiff'] - 2.5 * np.log10(df['flux_tot'])
    df['sigma_mag_tot'] = 1.0857 / df['SNR_tot']
    
    df['mag_diff'] = df['zpdiff'] - 2.5 * np.log10(df['forcediffimflux'].abs())
    df['sigma_mag_diff'] = 1.0857 / (df['forcediffimflux'].abs() / df['forcediffimfluxunc'])
    df['isdiffpos'] = int(1)
    df.loc[df['forcediffimflux'] < 0., 'isdiffpos'] = int(-1)
    
    filtermapping = {'ZTF_g': 1, 'ZTF_r': 2, 'ZTF_i': 3}
    df['fid'] = df['filter'].apply(lambda x: filtermapping[x]).astype(int)

    mask = df['dnearestrefsrc'] < DISTANCE_THRESHOLD
    cols = ['flux_tot', 'flux_tot_uJy',
            'fluxunc_tot', 'SNR_tot',
            'fluxunc_tot_uJy',
            'mag_tot', 'sigma_mag_tot']
    df.loc[~mask, cols] = np.nan
    
    return df

def correct_qid(df):
    df['qid'] = df['qid'] - 1
    return df

def clean_data_nanflux_infobits(df):
    # Filters the light curve data, keeping only the epochs with valid forcediffimflux values
    # and with infobitssci less than 33554432.
    
    # print('Filtering data to keep only epochs with valid difference flux and specific conditions on infobitssci.')
        
    mask = (df['forcediffimflux'].notna()) & (df['infobitssci'] < 33554432) 
    df = df[mask]
    
    return df

In [4]:
new_columns = [
    'index', 'mjd', 'nearestrefflux', 'flux_tot', 'flux_diff_uJy',
    'sigma_flux_diff_uJy', 'flux_tot_uJy', 'fluxunc_tot', 'SNR_tot',
    'fluxunc_tot_uJy', 'mag_tot', 'sigma_mag_tot', 'mag_diff',
    'sigma_mag_diff', 'isdiffpos', 'fid', 'secz', 'CCDquadID',
    'ZPthres', 'crit1', 'crit4', 'flag_bad', 'forcediffimfluxunc_resc',
    'sigma_flux_diff_uJy_resc', 'fluxunc_tot_resc',
    'fluxunc_tot_uJy_resc', 'SNR_tot_resc', 'sigma_mag_diff_resc',
    'sigma_mag_tot_resc', 'dnearestrefsrc', 'rfid', 'nearestrefchi', 
    'nearestrefsharp', 'detected']

In [5]:
path_chunks = glob.glob('detections/*')

# Dataset
detections = [] 
for i, path_chunk in enumerate(path_chunks):
    detections.append(pd.read_parquet(path_chunk))
    #if i == 2:
    #    break
detections = pd.concat(detections)
detections

ValueError: No objects to concatenate

In [6]:
detections.dtypes

index                     int64
field                     int64
ccdid                     int64
qid                       int64
filter                   object
pid                       int64
infobitssci               int64
sciinpseeing            float64
scibckgnd               float64
scisigpix               float64
zpmaginpsci             float64
zpmaginpsciunc          float64
zpmaginpscirms          float64
clrcoeff                float64
clrcoeffunc             float64
ncalmatches               int64
exptime                 float64
adpctdif1               float64
adpctdif2               float64
diffmaglim              float64
zpdiff                  float64
programid                 int64
jd                      float64
rfid                      int64
forcediffimflux         float64
forcediffimfluxunc      float64
forcediffimsnr          float64
forcediffimchisq        float64
forcediffimfluxap       float64
forcediffimfluxuncap    float64
forcediffimsnrap        float64
aperture

In [7]:
detections.index.nunique()

41809

In [8]:
df_first_last_mjd = pd.read_parquet('ts_v9.0.1_b3000/ts_v9.0.1_b3000_rangemjd.parquet').set_index('oid')
df_first_last_mjd

,survey_class_mapped,firstmjd,lastmjd
oid,,,
ZTF17aaazlzl,AGN,58363.482743,58900.187072
ZTF21acqoruu,AGN,59623.559595,60127.289074
ZTF22aaafnol,AGN,59671.362998,60085.221840
ZTF22aaaokdq,AGN,59623.400185,60021.358773
ZTF22aacbgva,AGN,59622.148692,60221.425683
...,...,...,...
ZTF18abmecii,ZZ,58435.112685,59870.330093
ZTF21aciipaz,ZZ,59504.531146,59504.531146
ZTF21aaotzvw,ZZ,59281.286887,59529.531007


In [9]:
groups_df = detections.groupby(level=0)
groups_df

In [10]:
del detections

In [11]:
processed_data = []
temporal_data = []

for oid, lightcurve in groups_df:
    # Procesar cada curva de luz aplicando todas las transformaciones necesarias
    lightcurve = (
        remove_negative_fluxunc(lightcurve)
        .pipe(clean_data_nanflux_infobits)
        .pipe(add_mjd)
        .pipe(add_detected)
    )

    # Filtrar curvas de luz con detecciones insuficientes
    if lightcurve['detected'].sum() < 2:
        continue

    # Corrección y recorte de datos
    lightcurve = correct_qid(lightcurve)
    lightcurve = limit_mjd(lightcurve, n_days_before=30, df_det=df_first_last_mjd.loc[oid])

    # Verificar si el DataFrame resultante está vacío
    if lightcurve.empty:
        continue

    processed_lightcurve = add_props(lightcurve)

    # Añadir a la lista temporal
    temporal_data.append(processed_lightcurve)

    # Si alcanzamos el tamaño del lote, concatenamos y vaciamos el lote temporal
    if len(temporal_data) >= 5000:
        processed_data.append(pd.concat(temporal_data))  # Concatenar lote
        temporal_data = []  # Reiniciar lote temporal

# Concatenar cualquier dato restante en el lote temporal
if temporal_data:
    processed_data.append(pd.concat(temporal_data))

del temporal_data

# Concatenar todos los bloques en un único DataFrame final
processed_data = pd.concat(processed_data)
processed_data

,index,field,ccdid,qid,filter,pid,infobitssci,sciinpseeing,scibckgnd,scisigpix,...,flux_tot_uJy,fluxunc_tot,SNR_tot,fluxunc_tot_uJy,mag_tot,sigma_mag_tot,mag_diff,sigma_mag_diff,isdiffpos,fid
oid,,,,,,,,,,,,,,,,,,,,,
ZTF17aaaaabj,56,742,8,3,ZTF_r,591472653115,0,1.7360,214.362,5.68734,...,2026.473988,30.769537,522.757259,3.876510,15.633147,0.002077,18.043294,0.019119,1,2
ZTF17aaaaabj,57,742,8,3,ZTF_r,592444453115,0,1.7424,206.338,5.89163,...,1440.802787,21.332058,529.287679,2.722154,16.003489,0.002051,17.492664,0.008085,-1,2
ZTF17aaaaabj,58,742,8,3,ZTF_g,592494673115,0,1.7377,130.065,4.59625,...,798.146432,17.936831,317.641224,2.512729,16.644794,0.003418,18.117736,0.013272,-1,1
ZTF17aaaaabj,59,742,8,3,ZTF_r,593450213115,0,1.4527,214.862,5.73343,...,1653.922182,20.062002,658.841280,2.510350,15.853712,0.001648,18.442393,0.017881,-1,2
ZTF17aaaaabj,60,742,8,3,ZTF_g,593491553115,0,1.8609,144.213,4.44819,...,803.451894,16.836916,376.580072,2.133549,16.637600,0.002883,18.146129,0.011568,-1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZTF23abpbuha,391,314,12,2,ZTF_r,2494483464615,0,3.9266,394.059,14.28720,...,NaN,NaN,NaN,NaN,NaN,NaN,18.327163,0.087250,1,2
ZTF23abpbuha,392,314,12,2,ZTF_g,2494509724615,0,4.4076,439.161,14.46490,...,NaN,NaN,NaN,NaN,NaN,NaN,20.127518,0.487193,1,1
ZTF23abpbuha,393,314,12,2,ZTF_r,2497481584615,0,3.8317,396.250,11.82860,...,NaN,NaN,NaN,NaN,NaN,NaN,18.409095,0.079053,1,2


In [12]:
del groups_df

In [13]:
# Filtrar las columnas que realmente existen en el DataFrame
existing_columns = [col for col in new_columns if col in processed_data.columns]

# Filtrar el DataFrame utilizando solo las columnas existentes
filtered_data = processed_data[existing_columns].reset_index()
filtered_data

,oid,index,mjd,nearestrefflux,flux_tot,flux_diff_uJy,sigma_flux_diff_uJy,flux_tot_uJy,fluxunc_tot,SNR_tot,...,sigma_mag_tot,mag_diff,sigma_mag_diff,isdiffpos,fid,dnearestrefsrc,rfid,nearestrefchi,nearestrefsharp,detected
0,ZTF17aaaaabj,56,58345.472651,14337.719489,16084.998820,220.131574,3.876510,2026.473988,30.769537,522.757259,...,0.002077,18.043294,0.019119,1,2,0.135209,742120231,0.455,-0.031,True
1,ZTF17aaaaabj,57,58346.444456,14155.332637,11290.795459,-365.539626,2.722154,1440.802787,21.332058,529.287679,...,0.002051,17.492664,0.008085,-1,2,0.135209,742120231,0.455,-0.031,True
2,ZTF17aaaaabj,58,58346.494676,7164.732169,5697.476945,-205.544407,2.512729,798.146432,17.936831,317.641224,...,0.003418,18.117736,0.013272,-1,1,0.141722,742120131,0.553,-0.057,True
3,ZTF17aaaaabj,59,58347.450220,14435.774167,13217.675084,-152.420231,2.510350,1653.922182,20.062002,658.841280,...,0.001648,18.442393,0.017881,-1,2,0.135209,742120231,0.455,-0.031,True
4,ZTF17aaaaabj,60,58347.491551,7920.634267,6340.447033,-200.238945,2.133549,803.451894,16.836916,376.580072,...,0.002883,18.146129,0.011568,-1,1,0.141722,742120131,0.553,-0.057,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53866294,ZTF23abpbuha,391,60248.483461,546.915262,NaN,169.486333,13.620362,NaN,NaN,NaN,...,NaN,18.327163,0.087250,1,2,1.548737,314120246,5.949,0.567,True
53866295,ZTF23abpbuha,392,60248.509722,316.548265,NaN,32.284393,14.487183,NaN,NaN,NaN,...,NaN,20.127518,0.487193,1,1,1.887613,314120146,4.107,0.280,False
53866296,ZTF23abpbuha,393,60251.481586,557.750596,NaN,157.167240,11.443844,NaN,NaN,NaN,...,NaN,18.409095,0.079053,1,2,1.548737,314120246,5.949,0.567,True
53866297,ZTF23abpbuha,394,60251.506343,322.403641,NaN,47.698061,6.817979,NaN,NaN,NaN,...,NaN,19.703748,0.155190,1,1,1.887613,314120146,4.107,0.280,True


In [14]:
filtered_data.index.nunique()

53866299

In [15]:
filtered_data.isnull().sum()

oid                          0
index                        0
mjd                          0
nearestrefflux          721662
flux_tot               1589023
flux_diff_uJy                0
sigma_flux_diff_uJy          0
flux_tot_uJy           1589023
fluxunc_tot            1589023
SNR_tot                1589023
fluxunc_tot_uJy        1589023
mag_tot                1834282
sigma_mag_tot          1589023
mag_diff                     0
sigma_mag_diff               0
isdiffpos                    0
fid                          0
dnearestrefsrc          721662
rfid                         0
nearestrefchi           721662
nearestrefsharp         721662
detected                     0
dtype: int64

In [16]:
filtered_data.dtypes

oid                     object
index                    int64
mjd                    float64
nearestrefflux         float64
flux_tot               float64
flux_diff_uJy          float64
sigma_flux_diff_uJy    float64
flux_tot_uJy           float64
fluxunc_tot            float64
SNR_tot                float64
fluxunc_tot_uJy        float64
mag_tot                float64
sigma_mag_tot          float64
mag_diff               float64
sigma_mag_diff         float64
isdiffpos                int64
fid                      int64
dnearestrefsrc         float64
rfid                     int64
nearestrefchi          float64
nearestrefsharp        float64
detected                  bool
dtype: object

In [17]:
import os
from datetime import datetime

# Obtener la fecha actual en formato YYMMDD
current_date = datetime.now().strftime('%y%m%d')
file_name = f'processed_{current_date}.parquet'
#file_name = 'processed_241015.parquet'

# Check if the file already exists before saving
if not os.path.exists(file_name):
    filtered_data.to_parquet(file_name)
    print(f"File '{file_name}' successfully saved.")
else:
    print(f"The file '{file_name}' already exists. It won't be saved again.")


File 'processed_241015.parquet' successfully saved.


In [1]:
import pandas as pd

filtered_data = pd.read_parquet('processed_241119.parquet')
filtered_data

,oid,index,mjd,nearestrefflux,flux_tot,flux_diff_uJy,sigma_flux_diff_uJy,flux_tot_uJy,fluxunc_tot,SNR_tot,...,sigma_mag_tot,mag_diff,sigma_mag_diff,isdiffpos,fid,dnearestrefsrc,rfid,nearestrefchi,nearestrefsharp,detected
0,ZTF17aaaaabj,56,58345.472651,14337.719489,16084.998820,220.131574,3.876510,2026.473988,30.769537,522.757259,...,0.002077,18.043294,0.019119,1,2,0.135209,742120231,0.455,-0.031,True
1,ZTF17aaaaabj,57,58346.444456,14155.332637,11290.795459,-365.539626,2.722154,1440.802787,21.332058,529.287679,...,0.002051,17.492664,0.008085,-1,2,0.135209,742120231,0.455,-0.031,True
2,ZTF17aaaaabj,58,58346.494676,7164.732169,5697.476945,-205.544407,2.512729,798.146432,17.936831,317.641224,...,0.003418,18.117736,0.013272,-1,1,0.141722,742120131,0.553,-0.057,True
3,ZTF17aaaaabj,59,58347.450220,14435.774167,13217.675084,-152.420231,2.510350,1653.922182,20.062002,658.841280,...,0.001648,18.442393,0.017881,-1,2,0.135209,742120231,0.455,-0.031,True
4,ZTF17aaaaabj,60,58347.491551,7920.634267,6340.447033,-200.238945,2.133549,803.451894,16.836916,376.580072,...,0.002883,18.146129,0.011568,-1,1,0.141722,742120131,0.553,-0.057,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53866294,ZTF23abpbuha,391,60248.483461,546.915262,NaN,169.486333,13.620362,NaN,NaN,NaN,...,NaN,18.327163,0.087250,1,2,1.548737,314120246,5.949,0.567,True
53866295,ZTF23abpbuha,392,60248.509722,316.548265,NaN,32.284393,14.487183,NaN,NaN,NaN,...,NaN,20.127518,0.487193,1,1,1.887613,314120146,4.107,0.280,False
53866296,ZTF23abpbuha,393,60251.481586,557.750596,NaN,157.167240,11.443844,NaN,NaN,NaN,...,NaN,18.409095,0.079053,1,2,1.548737,314120246,5.949,0.567,True
53866297,ZTF23abpbuha,394,60251.506343,322.403641,NaN,47.698061,6.817979,NaN,NaN,NaN,...,NaN,19.703748,0.155190,1,1,1.887613,314120146,4.107,0.280,True


In [4]:
import numpy as np

# Calcular estadísticas para cada banda
stats = filtered_data.groupby('fid')['flux_diff_uJy'].agg(
    Mean=('mean'),
    Std=('std'),
    Min=('min'),
    P25=lambda x: np.percentile(x, 25),
    Median=('median'),
    P75=lambda x: np.percentile(x, 75),
    Max=('max')
).round(4)

# Mostrar resultados
stats.reset_index(inplace=True)  # Opcional: si quieres mantener 'fid' como columna
print(stats)

   fid     Mean        Std           Min       P25  Median       P75  \
0    1  57.7924  2275.2261 -2.136146e+06  -80.8610 -0.8780   83.9467   
1    2  45.6172  3127.8604 -9.535759e+04 -126.8360 -0.0038  128.5438   
2    3 -32.5167  1980.8888 -5.419769e+04  -97.5573 -3.6380   81.3004   

            Max  
0  1.605385e+06  
1  1.044464e+06  
2  2.745450e+05  


In [5]:
stats

,fid,Mean,Std,Min,P25,Median,P75,Max
0,1,57.7924,2275.2261,-2.136146e+06,-80.8610,-0.8780,83.9467,1.605385e+06
1,2,45.6172,3127.8604,-9.535759e+04,-126.8360,-0.0038,128.5438,1.044464e+06
2,3,-32.5167,1980.8888,-5.419769e+04,-97.5573,-3.6380,81.3004,2.745450e+05


In [ ]:
filtered_data

In [21]:
filtered_data.isnull().sum()

oid                          0
index                        0
mjd                          0
nearestrefflux          721662
flux_tot               1589023
flux_diff_uJy                0
sigma_flux_diff_uJy          0
flux_tot_uJy           1589023
fluxunc_tot            1589023
SNR_tot                1589023
fluxunc_tot_uJy        1589023
mag_tot                1834282
sigma_mag_tot          1589023
mag_diff                     0
sigma_mag_diff               0
isdiffpos                    0
fid                          0
dnearestrefsrc          721662
rfid                         0
nearestrefchi           721662
nearestrefsharp         721662
detected                     0
dtype: int64

## If you want to insert in the DB

In [22]:
import os

table_name = file_name.split('.')[0]
os.makedirs(f'parquets/{table_name}', exist_ok=True)

In [23]:
!ln -s ~/data_ZTF_ff/{file_name} ~/data_ZTF_ff/parquets/{table_name}/

In [24]:
filtered_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53866299 entries, 0 to 53866298
Data columns (total 22 columns):
 #   Column               Dtype  
---  ------               -----  
 0   oid                  object 
 1   index                int64  
 2   mjd                  float64
 3   nearestrefflux       float64
 4   flux_tot             float64
 5   flux_diff_uJy        float64
 6   sigma_flux_diff_uJy  float64
 7   flux_tot_uJy         float64
 8   fluxunc_tot          float64
 9   SNR_tot              float64
 10  fluxunc_tot_uJy      float64
 11  mag_tot              float64
 12  sigma_mag_tot        float64
 13  mag_diff             float64
 14  sigma_mag_diff       float64
 15  isdiffpos            int64  
 16  fid                  int64  
 17  dnearestrefsrc       float64
 18  rfid                 int64  
 19  nearestrefchi        float64
 20  nearestrefsharp      float64
 21  detected             bool   
dtypes: bool(1), float64(16), int64(4), object(1)
memory usage: 8.5+ 

In [25]:
def generate_create_table_sql(df, table_name):
    # Envolver el nombre de la tabla en comillas dobles
    sql = f'CREATE TABLE "{table_name}" (\n'
    for col in df.columns:
        col = f'"{col}"'  # Envolver el nombre de la columna en comillas dobles
        dtype = df[col.strip('"')].dtype  # Obtener el tipo de dato de la columna original

        # Mapear el tipo de dato de Pandas a PostgreSQL
        if dtype == 'int64':
            sql += f"    {col} BIGINT,\n"
        elif dtype == 'float64':
            sql += f"    {col} DOUBLE PRECISION,\n"
        elif dtype == 'object':
            sql += f"    {col} TEXT,\n"
        elif 'datetime' in str(dtype):
            sql += f"    {col} TIMESTAMP,\n"
        elif dtype == 'bool':
            sql += f"    {col} BOOLEAN,\n"
        else:
            sql += f"    {col} TEXT,\n"  # Tipo de dato por defecto

    sql = sql.rstrip(",\n") + "\n);"
    return sql

# Generar el comando SQL
create_table_sql = generate_create_table_sql(filtered_data, table_name)
print(create_table_sql)


CREATE TABLE "processed_241119" (
    "oid" TEXT,
    "index" BIGINT,
    "mjd" DOUBLE PRECISION,
    "nearestrefflux" DOUBLE PRECISION,
    "flux_tot" DOUBLE PRECISION,
    "flux_diff_uJy" DOUBLE PRECISION,
    "sigma_flux_diff_uJy" DOUBLE PRECISION,
    "flux_tot_uJy" DOUBLE PRECISION,
    "fluxunc_tot" DOUBLE PRECISION,
    "SNR_tot" DOUBLE PRECISION,
    "fluxunc_tot_uJy" DOUBLE PRECISION,
    "mag_tot" DOUBLE PRECISION,
    "sigma_mag_tot" DOUBLE PRECISION,
    "mag_diff" DOUBLE PRECISION,
    "sigma_mag_diff" DOUBLE PRECISION,
    "isdiffpos" BIGINT,
    "fid" BIGINT,
    "dnearestrefsrc" DOUBLE PRECISION,
    "rfid" BIGINT,
    "nearestrefchi" DOUBLE PRECISION,
    "nearestrefsharp" DOUBLE PRECISION,
    "detected" BOOLEAN
);


In [26]:
import psycopg2
import json

# Conectar a la base de datos
with open('cosmicstreams_credentials.json') as jsonfile:
    db_credentials = json.load(jsonfile)

conn = psycopg2.connect(
    host=db_credentials['host'],
    port=db_credentials['port'],
    user=db_credentials['user'],
    password=db_credentials['password'],
    database=db_credentials['database']
)

cur = conn.cursor()
cur.execute(create_table_sql)
conn.commit()
cur.close()
conn.close()

In [27]:
conn = psycopg2.connect(
    host=db_credentials['host'],
    port=db_credentials['port'],
    user=db_credentials['user'],
    password=db_credentials['password'],
    database=db_credentials['database']
)

# Crear un cursor
cur = conn.cursor()

data_check = pd.read_sql(f"SELECT * FROM {file_name.split('.')[0]};", conn)

cur.close()
conn.close()

/tmp/ipykernel_2330380/3311928401.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_check = pd.read_sql(f"SELECT * FROM {file_name.split('.')[0]};", conn)


In [28]:
data_check

,oid,index,mjd,nearestrefflux,flux_tot,flux_diff_uJy,sigma_flux_diff_uJy,flux_tot_uJy,fluxunc_tot,SNR_tot,...,sigma_mag_tot,mag_diff,sigma_mag_diff,isdiffpos,fid,dnearestrefsrc,rfid,nearestrefchi,nearestrefsharp,detected


In [29]:
import yaml

def generate_mapping_yaml(df, parquet_name, table_name, output_file="mapping.yml"):
    # Crear el diccionario para el mapeo
    mapping = {
        "tables": [
            {
                "from": parquet_name,
                "to": table_name,
                "columns": [
                    {"from": col, "to": col} for col in df.columns
                ]
            }
        ]
    }

    # Guardar el archivo mapping.yml
    with open(output_file, "w") as f:
        yaml.dump(mapping, f, default_flow_style=False, sort_keys=False)

    print(f"Archivo '{output_file}' generado exitosamente.")

# Parámetros
parquet_name = table_name
table_name = table_name

# Generar el archivo mapping.yml
generate_mapping_yaml(filtered_data, parquet_name, table_name)

Archivo 'mapping.yml' generado exitosamente.


In [30]:
import toml

def generate_mapping_toml(df, parquet_name, table_name, output_file="mapping.toml"):
    # Crear el diccionario para el mapeo
    mapping = {
        "tables": [
            {
                "from": parquet_name,
                "to": table_name,
                "columns": [
                    {"from": col, "to": col} for col in df.columns
                ]
            }
        ]
    }

    # Guardar el archivo mapping.toml
    with open(output_file, "w") as f:
        toml.dump(mapping, f)

    print(f"Archivo '{output_file}' generado exitosamente.")

# Parámetros proporcionados
parquet_name = table_name
table_name = table_name

# Generar el archivo mapping.toml
generate_mapping_toml(filtered_data, parquet_name, table_name)

Archivo 'mapping.toml' generado exitosamente.
